# 02 — Windowing and train/test split

Reads `cleaned_data/`, slices 2 s windows (50% overlap).

- **normal walking**: train = `session_1` + `session_2`, test = `session_3`
- **crowded walking**: train = `session_1`, test = `session_2`

Output: `windowed_data/`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path('.').resolve()
CLEAN = ROOT / 'cleaned_data'
OUT = ROOT / 'windowed_data'
OUT.mkdir(exist_ok=True)

WINDOW_S = 2.0
STEP_S = 1.0   # 50% overlap for 2 s windows
HZ = 100
SENSOR_COLS = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']

manifest = pd.read_csv(CLEAN / 'manifest.csv')

In [2]:
def extract_windows(df, window_s=WINDOW_S, step_s=STEP_S, hz=HZ):
    win_len = int(window_s * hz)
    step = int(step_s * hz)
    data = df[SENSOR_COLS].to_numpy()
    return np.array([data[i:i + win_len] for i in range(0, len(data) - win_len + 1, step)])


def assign_split(context, session):
    if context == 'crowded':
        return 'test' if session == 'session_2' else 'train'
    return 'test' if session == 'session_3' else 'train'


def build_window_tables(records):
    meta_rows = []
    arrays = []
    for _, rec in records.iterrows():
        path = CLEAN / rec['participant'] / rec['context'] / rec['session'] / 'Gait Data.csv'
        X = extract_windows(pd.read_csv(path))
        if len(X) == 0:
            continue
        split = assign_split(rec['context'], rec['session'])
        arrays.append(X)
        for i in range(len(X)):
            meta_rows.append({
                'participant': rec['participant'],
                'context': rec['context'],
                'session': rec['session'],
                'split': split,
                'window_idx': i,
                'label_person': rec['participant'],
                'label_context': rec['context'],
            })
    X_all = np.concatenate(arrays, axis=0) if arrays else np.empty((0, int(WINDOW_S * HZ), 6))
    return pd.DataFrame(meta_rows), X_all


meta, X = build_window_tables(manifest)
meta.head()

,participant,context,session,split,window_idx,label_person,label_context
0,Ana,crowded,session_1,train,0,Ana,crowded
1,Ana,crowded,session_1,train,1,Ana,crowded
2,Ana,crowded,session_1,train,2,Ana,crowded
3,Ana,crowded,session_1,train,3,Ana,crowded
4,Ana,crowded,session_1,train,4,Ana,crowded


In [3]:
# save full window set
meta_train = meta[meta['split'] == 'train'].reset_index(drop=True)
meta_test = meta[meta['split'] == 'test'].reset_index(drop=True)
train_idx = meta.index[meta['split'] == 'train'].to_numpy()
test_idx = meta.index[meta['split'] == 'test'].to_numpy()

X_train, X_test = X[train_idx], X[test_idx]
np.save(OUT / 'X_train.npy', X_train)
np.save(OUT / 'X_test.npy', X_test)
meta_train.to_csv(OUT / 'meta_train.csv', index=False)
meta_test.to_csv(OUT / 'meta_test.csv', index=False)

print('all windows — train:', X_train.shape, 'test:', X_test.shape)
meta.groupby(['split', 'context']).size()

all windows — train: (2498, 200, 6) test: (1770, 200, 6)


split  context
test   crowded    1054
       normal      716
train  crowded    1066
       normal     1432
dtype: int64

In [4]:
# normal-walking cross-session subset (main person-ID / auth task)
eligible = set(manifest.loc[manifest['cross_session_eligible'], 'participant'].unique())
normal = meta[(meta['context'] == 'normal') & (meta['participant'].isin(eligible))]
idx = normal.index.to_numpy()
X_normal = X[idx]
normal = normal.reset_index(drop=True)
normal['array_idx'] = range(len(normal))

n_train = normal[normal['split'] == 'train']
n_test = normal[normal['split'] == 'test']
X_normal_train = X_normal[n_train['array_idx'].to_numpy()]
X_normal_test = X_normal[n_test['array_idx'].to_numpy()]

np.save(OUT / 'X_normal_train.npy', X_normal_train)
np.save(OUT / 'X_normal_test.npy', X_normal_test)
n_train.drop(columns='array_idx').to_csv(OUT / 'normal_meta_train.csv', index=False)
n_test.drop(columns='array_idx').to_csv(OUT / 'normal_meta_test.csv', index=False)

print('normal cross-session — train:', X_normal_train.shape, 'test:', X_normal_test.shape)
print('participants:', n_train.participant.nunique())

normal cross-session — train: (1432, 200, 6) test: (716, 200, 6)
participants: 4
